# Week 9: Gradient Boosting on My Two Datasets

For this week, I wanted to try gradient boosting on both of the supervised datasets I have been using in this project:

1. Wisconsin breast cancer diagnosis
2. Brazil patient outcome dataset

The main thing I wanted to see was how gradient boosting compares to random forest on each one. I also wanted to keep this notebook pretty straightforward, so I am doing the model fitting and graphs directly in the notebook instead of hiding a lot of the work in helper functions.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.load_data import load_wisconsin, load_brazil, split_X_y

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)


## 1. Wisconsin Dataset

I started with Wisconsin because it is smaller and cleaner, which makes it a good dataset for testing the full gradient boosting workflow.

In this section, I am checking the class balance, splitting the data into training and test sets, tuning a few gradient boosting settings, and then comparing the results to random forest.


In [ ]:
wisconsin_df, wisconsin_target = load_wisconsin()
X_w, y_w = split_X_y(wisconsin_df, wisconsin_target)

print("Wisconsin shape:", wisconsin_df.shape)
display(y_w.value_counts().rename("count").to_frame())
display((y_w.value_counts(normalize=True).rename("share") * 100).round(2).to_frame())

X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_w, y_w, test_size=0.25, random_state=42, stratify=y_w
)

print("Training rows:", X_w_train.shape[0])
print("Test rows:", X_w_test.shape[0])


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

wisconsin_rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
wisconsin_rf.fit(X_w_train, y_w_train)

wisconsin_gb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [1, 2, 3],
    "subsample": [0.8, 1.0],
}

wisconsin_gb_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=wisconsin_gb_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
)
wisconsin_gb_search.fit(X_w_train, y_w_train)

wisconsin_gb = wisconsin_gb_search.best_estimator_
print("Best Wisconsin gradient boosting parameters:")
print(wisconsin_gb_search.best_params_)


In [ ]:
wisconsin_results = []

for model_name, model in {
    "Random Forest": wisconsin_rf,
    "Gradient Boosting": wisconsin_gb,
}.items():
    y_pred = model.predict(X_w_test)
    y_score = model.predict_proba(X_w_test)[:, 1]

    wisconsin_results.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_w_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_w_test, y_pred),
            "f1": f1_score(y_w_test, y_pred),
            "roc_auc": roc_auc_score(y_w_test, y_score),
            "average_precision": average_precision_score(y_w_test, y_score),
        }
    )

wisconsin_results_df = pd.DataFrame(wisconsin_results).sort_values("roc_auc", ascending=False)
display(wisconsin_results_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for model_name, model, color in [
    ("Random Forest", wisconsin_rf, "#3b82f6"),
    ("Gradient Boosting", wisconsin_gb, "#ef4444"),
]:
    y_pred = model.predict(X_w_test)
    y_score = model.predict_proba(X_w_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_w_test, y_score)
    precision, recall, _ = precision_recall_curve(y_w_test, y_score)

    axes[0].plot(fpr, tpr, label=model_name, color=color, linewidth=2)
    axes[1].plot(recall, precision, label=model_name, color=color, linewidth=2)

cm = confusion_matrix(y_w_test, wisconsin_gb.predict(X_w_test))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", cbar=False, ax=axes[2])

axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("Wisconsin ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].set_title("Wisconsin Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

axes[2].set_title("Wisconsin Confusion Matrix\nGradient Boosting")
axes[2].set_xlabel("Predicted label")
axes[2].set_ylabel("True label")

plt.tight_layout()
plt.show()

print("Wisconsin classification report for gradient boosting:")
print(classification_report(y_w_test, wisconsin_gb.predict(X_w_test)))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(wisconsin_gb.train_score_, color="#ef4444", linewidth=2)
ax.set_title("Wisconsin Gradient Boosting Training Deviance")
ax.set_xlabel("Boosting iteration")
ax.set_ylabel("Training deviance")
plt.tight_layout()
plt.show()


In [ ]:
wisconsin_perm = permutation_importance(
    wisconsin_gb,
    X_w_test,
    y_w_test,
    n_repeats=20,
    random_state=42,
    scoring="roc_auc",
)

wisconsin_importance = (
    pd.DataFrame(
        {
            "feature": X_w.columns,
            "importance": wisconsin_perm.importances_mean,
        }
    )
    .sort_values("importance", ascending=False)
    .head(10)
)

plt.figure(figsize=(9, 5))
sns.barplot(data=wisconsin_importance, x="importance", y="feature", color="#ef4444")
plt.title("Wisconsin Top 10 Features by Permutation Importance")
plt.xlabel("Mean decrease in ROC AUC")
plt.ylabel("")
plt.tight_layout()
plt.show()

wisconsin_top_features = wisconsin_importance["feature"].head(2).tolist()
PartialDependenceDisplay.from_estimator(
    wisconsin_gb,
    X_w_train,
    features=wisconsin_top_features,
)
plt.suptitle("Wisconsin Partial Dependence Plots", y=1.02)
plt.tight_layout()
plt.show()


### Wisconsin Takeaway

For Wisconsin, regular gradient boosting works well because the dataset is small enough that I can tune it without making the workflow too heavy.

The graphs that help the most here are the ROC curve, precision-recall curve, confusion matrix, training deviance over the boosting iterations, and the feature interpretation plots.


## 2. Brazil Dataset

The Brazil dataset needed a different approach because it is much larger and the target is more imbalanced.

Instead of copying the exact same setup from Wisconsin, I used a more practical workflow: sample a manageable number of rows, keep a random forest baseline, use `HistGradientBoostingClassifier`, and pay more attention to ROC AUC, average precision, and threshold tradeoffs.


In [ ]:
brazil_sample_size = 120_000

brazil_df, brazil_target = load_brazil(sample_size=brazil_sample_size, random_state=42, rename_cols=True)
X_b, y_b = split_X_y(brazil_df, brazil_target)

print("Brazil sample shape:", brazil_df.shape)
display(y_b.value_counts().rename("count").to_frame())
display((y_b.value_counts(normalize=True).rename("share") * 100).round(2).to_frame())

X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_b, y_b, test_size=0.25, random_state=42, stratify=y_b
)

print("Training rows:", X_b_train.shape[0])
print("Test rows:", X_b_test.shape[0])


In [ ]:
brazil_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
brazil_rf.fit(X_b_train, y_b_train)

brazil_hgb_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [3, 5, None],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [50, 200],
    "l2_regularization": [0.0, 1.0],
}

brazil_hgb_search = GridSearchCV(
    estimator=HistGradientBoostingClassifier(
        max_iter=250,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    ),
    param_grid=brazil_hgb_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
)
brazil_hgb_search.fit(X_b_train, y_b_train)

brazil_hgb = brazil_hgb_search.best_estimator_
print("Best Brazil HistGradientBoosting parameters:")
print(brazil_hgb_search.best_params_)


In [ ]:
brazil_results = []

for model_name, model in {
    "Random Forest": brazil_rf,
    "HistGradientBoosting": brazil_hgb,
}.items():
    y_pred = model.predict(X_b_test)
    y_score = model.predict_proba(X_b_test)[:, 1]

    brazil_results.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_b_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_b_test, y_pred),
            "f1": f1_score(y_b_test, y_pred),
            "roc_auc": roc_auc_score(y_b_test, y_score),
            "average_precision": average_precision_score(y_b_test, y_score),
        }
    )

brazil_results_df = pd.DataFrame(brazil_results).sort_values("average_precision", ascending=False)
display(brazil_results_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for model_name, model, color in [
    ("Random Forest", brazil_rf, "#2563eb"),
    ("HistGradientBoosting", brazil_hgb, "#dc2626"),
]:
    y_score = model.predict_proba(X_b_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_b_test, y_score)
    precision, recall, _ = precision_recall_curve(y_b_test, y_score)

    axes[0].plot(fpr, tpr, label=model_name, color=color, linewidth=2)
    axes[1].plot(recall, precision, label=model_name, color=color, linewidth=2)

cm = confusion_matrix(y_b_test, brazil_hgb.predict(X_b_test))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", cbar=False, ax=axes[2])

axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("Brazil ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].set_title("Brazil Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

axes[2].set_title("Brazil Confusion Matrix\nHistGradientBoosting")
axes[2].set_xlabel("Predicted label")
axes[2].set_ylabel("True label")

plt.tight_layout()
plt.show()

print("Brazil classification report for HistGradientBoosting:")
print(classification_report(y_b_test, brazil_hgb.predict(X_b_test)))


In [ ]:
brazil_scores = brazil_hgb.predict_proba(X_b_test)[:, 1]
thresholds = np.linspace(0.05, 0.50, 19)

threshold_rows = []
for threshold in thresholds:
    threshold_pred = (brazil_scores >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": threshold,
            "balanced_accuracy": balanced_accuracy_score(y_b_test, threshold_pred),
            "f1": f1_score(y_b_test, threshold_pred),
        }
    )

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(4))

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["balanced_accuracy"], label="Balanced accuracy", linewidth=2)
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1 score", linewidth=2)
plt.title("Brazil Threshold Tuning for HistGradientBoosting")
plt.xlabel("Classification threshold")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
brazil_sample_idx = X_b_test.sample(n=5000, random_state=42).index

brazil_perm = permutation_importance(
    brazil_hgb,
    X_b_test.loc[brazil_sample_idx],
    y_b_test.loc[brazil_sample_idx],
    n_repeats=10,
    random_state=42,
    scoring="average_precision",
)

brazil_importance = (
    pd.DataFrame(
        {
            "feature": X_b.columns,
            "importance": brazil_perm.importances_mean,
        }
    )
    .sort_values("importance", ascending=False)
    .head(12)
)

plt.figure(figsize=(9, 6))
sns.barplot(data=brazil_importance, x="importance", y="feature", color="#dc2626")
plt.title("Brazil Top Features by Permutation Importance")
plt.xlabel("Mean decrease in average precision")
plt.ylabel("")
plt.tight_layout()
plt.show()


### Brazil Takeaway

For Brazil, gradient boosting still makes sense, but I had to adjust the workflow to match the size and balance of the data.

The biggest points here are using the histogram-based version for speed, evaluating the model with class imbalance in mind, looking closely at the precision-recall curve, and checking whether a different classification threshold works better than `0.50`.


## Final Comparison Notes

What stood out to me is that the same overall idea worked on both datasets, but the implementation needed to change depending on the data.

- **Wisconsin:** I could use regular gradient boosting and spend more time on interpretation.
- **Brazil:** I needed a more efficient model setup and more careful evaluation because of the dataset size and class imbalance.

The main points I would highlight from this notebook are:

1. Gradient boosting builds trees one at a time, with each new tree trying to improve the earlier model.
2. Smaller learning rates usually mean the model needs more boosting rounds.
3. Depth, leaf size, and related settings matter because they control how easily the model can overfit.
4. For imbalanced data, accuracy by itself is not enough, so precision-recall metrics matter more.
5. Even when the method stays the same, the workflow should match the dataset you are working with.
